# 03d. SMOTE + Feature Selection Trials

**Project:** KPI-RAG: Explainable Root-Cause Analysis for 5G Networks
**Stage:** Phase 1 — Advanced Feature Trials

## Purpose

Notebook 03c confirmed Config A (64-dim) dominates all 4 algorithms.
Root cause: 988 anomalous samples / 582 features = 1.7 samples per feature.

This notebook runs two targeted trials:

**Trial 1 — SMOTE Oversampling**
Synthetically increases anomalous training samples from 988 to ~5,000.
If successful, Config B and C should improve and may surpass Config A.

**Trial 2 — SelectKBest Feature Selection**
Selects the best 64 features from the full 582 using statistical tests.
If the best 64 selected features outperform the first 64 (Config A),
it proves that better feature selection beats hand-crafted ordering.

## Key Question
Does the performance gap between configs disappear when we have more samples or better feature selection?

## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import os
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

COLORS = {
    'normal':     '#2E6DB4',
    'anomaly':    '#C0392B',
    'jamming':    '#E67E22',
    'synthetic':  '#85929E',
    'axis':       '#2C3E50',
    'grid':       '#F2F4F6',
}

DATA_DIR = r'C:\Users\DELL\Desktop\kpi_rag\data'

X_full       = np.load(os.path.join(DATA_DIR, 'X_features.npy'))
y_binary     = np.load(os.path.join(DATA_DIR, 'y_binary.npy'))
y_multiclass = np.load(os.path.join(DATA_DIR, 'y_multiclass.npy'))
train_idx    = np.load(os.path.join(DATA_DIR, 'train_idx.npy'))
test_idx     = np.load(os.path.join(DATA_DIR, 'test_idx.npy'))

with open(os.path.join(DATA_DIR, 'type_to_int.json')) as f:
    type_to_int = json.load(f)

X_A = X_full[:, :64]
X_B = X_full[:, :320]
X_C = X_full[:, :582]

X_A_train, X_A_test = X_A[train_idx], X_A[test_idx]
X_B_train, X_B_test = X_B[train_idx], X_B[test_idx]
X_C_train, X_C_test = X_C[train_idx], X_C[test_idx]

y_bin_train = y_binary[train_idx]
y_bin_test  = y_binary[test_idx]
y_mc_train  = y_multiclass[train_idx]
y_mc_test   = y_multiclass[test_idx]

RF_PARAMS = {
    'n_estimators':  300,
    'max_depth':     None,
    'min_samples_leaf': 2,
    'n_jobs':        4,
    'random_state':  42,
    'class_weight':  'balanced',
}

try:
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    BOOSTING_AVAILABLE = True
    print('XGBoost and LightGBM available.')
except ImportError:
    BOOSTING_AVAILABLE = False
    print('XGBoost/LightGBM not available — using Random Forest only.')

print(f'Train: {len(train_idx):,}   Test: {len(test_idx):,}')
print(f'Train anomalies: {y_bin_train.sum():,}')

## 2. Baseline — No SMOTE (Reference from Notebook 03c)

These are the reference results from Notebook 03c for comparison.

In [ ]:
BASELINE = {
    'binary': {
        'A': {'RF': 0.9835, 'LightGBM': 0.9856},
        'B': {'RF': 0.9708, 'XGBoost':  0.9836},
        'C': {'RF': 0.9665, 'LightGBM': 0.9835},
    },
    'fault': {
        'A': {'RF': 0.9447, 'XGBoost':  0.9564},
        'B': {'RF': 0.8866, 'LightGBM': 0.9404},
        'C': {'RF': 0.8872, 'LightGBM': 0.9045},
    }
}

print('Baseline (No SMOTE) from Notebook 03c:')
print(f'  Best Binary F1:  LightGBM + Config A = 0.9856')
print(f'  Best Fault F1:   XGBoost  + Config A = 0.9564')
print()
print('Training set composition:')
print(f'  Normal:  {(y_bin_train == 0).sum():,}')
print(f'  Anomaly: {(y_bin_train == 1).sum():,}')
print(f'  Ratio:   {(y_bin_train == 1).sum() / len(y_bin_train) * 100:.1f}% anomaly')

## 3. Trial 1 — SMOTE Oversampling

SMOTE (Synthetic Minority Oversampling Technique) creates synthetic anomalous samples
by interpolating between existing anomalous samples in feature space.

**Target:** Increase anomalous training samples from 988 to ~4,000 (sampling_strategy=0.2
means anomaly will be 20% of the majority class count).

**Applied to:** Training set only — test set stays untouched.
This is critical: SMOTE on test set would inflate metrics artificially.

In [ ]:
print('Applying SMOTE to training sets...')
print(f'Before: Normal={( y_bin_train==0).sum():,}  Anomaly={(y_bin_train==1).sum():,}')

smote = SMOTE(sampling_strategy=0.2, random_state=42, n_jobs=4)

X_A_train_sm, y_bin_train_sm = smote.fit_resample(X_A_train, y_bin_train)
X_B_train_sm, _              = smote.fit_resample(X_B_train, y_bin_train)
X_C_train_sm, _              = smote.fit_resample(X_C_train, y_bin_train)

print(f'After:  Normal={(y_bin_train_sm==0).sum():,}  Anomaly={(y_bin_train_sm==1).sum():,}')
print(f'Samples per feature (Config C): {(y_bin_train_sm==1).sum() / 582:.1f}')
print()
print('Note: Test sets are NOT resampled — evaluation remains on natural distribution.')

## 4. SMOTE Binary Detection Results

In [ ]:
def eval_binary(model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return {
        'f1':        f1_score(y_te, y_pred, pos_label=1),
        'precision': precision_score(y_te, y_pred, pos_label=1, zero_division=0),
        'recall':    recall_score(y_te, y_pred, pos_label=1, zero_division=0),
    }

print('SMOTE Binary Detection Results:')
print(f'{"Config":<12} {"RF F1":>8} {"vs baseline":>12}')
print('-' * 36)

smote_binary_rf = {}
for config_name, X_tr, X_te, baseline_f1 in [
    ('A (64-dim)',  X_A_train_sm, X_A_test, 0.9835),
    ('B (320-dim)', X_B_train_sm, X_B_test, 0.9708),
    ('C (582-dim)', X_C_train_sm, X_C_test, 0.9665),
]:
    rf = RandomForestClassifier(**RF_PARAMS)
    result = eval_binary(rf, X_tr, X_te, y_bin_train_sm, y_bin_test)
    change = result['f1'] - baseline_f1
    arrow = '+' if change >= 0 else ''
    print(f'{config_name:<12} {result["f1"]:>8.4f} {arrow}{change:>10.4f}')
    smote_binary_rf[config_name] = result

## 5. SMOTE Fault Classifier Results

In [ ]:
def eval_multiclass_smote(X_train, X_test, y_train, y_test, jamming_class):
    mask_tr = (y_train > 0) & (y_train != jamming_class)
    mask_te = (y_test  > 0) & (y_test  != jamming_class)
    X_tr, y_tr = X_train[mask_tr], y_train[mask_tr]
    X_te, y_te = X_test[mask_te],  y_test[mask_te]

    # SMOTE on fault classifier training data
    smote_mc = SMOTE(sampling_strategy='not majority', random_state=42, n_jobs=4)
    try:
        X_tr_sm, y_tr_sm = smote_mc.fit_resample(X_tr, y_tr)
    except Exception:
        X_tr_sm, y_tr_sm = X_tr, y_tr

    unique = np.sort(np.unique(y_tr_sm))
    c2i = {c: i for i, c in enumerate(unique)}
    y_tr_m = np.array([c2i[c] for c in y_tr_sm])
    y_te_m = np.array([c2i.get(c, -1) for c in y_te])
    valid  = y_te_m >= 0
    X_te, y_te_m = X_te[valid], y_te_m[valid]

    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_tr_sm, y_tr_m)
    y_pred = rf.predict(X_te)
    return {
        'f1_macro': f1_score(y_te_m, y_pred, average='macro', zero_division=0),
        'accuracy': (y_pred == y_te_m).mean(),
        'n_train':  len(X_tr_sm),
    }

jamming_class = type_to_int.get('Jamming', -1)

# We need full 582-dim multiclass labels aligned with SMOTE — use original (no smote on mc labels)
print('SMOTE Fault Classifier Results (RF):')
print(f'{"Config":<12} {"F1-macro":>10} {"vs baseline":>12} {"Train samples":>14}')
print('-' * 52)

smote_mc_rf = {}
for config_name, X_tr, X_te, baseline_f1 in [
    ('A (64-dim)',  X_A_train, X_A_test, 0.9447),
    ('B (320-dim)', X_B_train, X_B_test, 0.8866),
    ('C (582-dim)', X_C_train, X_C_test, 0.8872),
]:
    result = eval_multiclass_smote(X_tr, X_te, y_mc_train, y_mc_test, jamming_class)
    change = result['f1_macro'] - baseline_f1
    arrow = '+' if change >= 0 else ''
    print(f'{config_name:<12} {result["f1_macro"]:>10.4f} {arrow}{change:>10.4f} {result["n_train"]:>14,}')
    smote_mc_rf[config_name] = result

## 6. Trial 2 — SelectKBest Feature Selection

Instead of taking the first 64 features (Config A), we select the best 64 features
from the full 582 using ANOVA F-test (f_classif).

This answers: "Are the first 64 features actually the best 64?" 

In [ ]:
print('Running SelectKBest — selecting best 64 features from 582...')

selector_binary = SelectKBest(f_classif, k=64)
selector_binary.fit(X_C_train, y_bin_train)
X_kbest_train = selector_binary.transform(X_C_train)
X_kbest_test  = selector_binary.transform(X_C_test)

selected_indices = selector_binary.get_support(indices=True)
print(f'Selected feature indices (first 10): {selected_indices[:10]}')
print(f'How many from stats block [0:64]:    {(selected_indices < 64).sum()}')
print(f'How many from scale block [64:320]:  {((selected_indices >= 64) & (selected_indices < 320)).sum()}')
print(f'How many from diffs block [320:576]: {((selected_indices >= 320) & (selected_indices < 576)).sum()}')
print(f'How many from cats block [576:582]:  {(selected_indices >= 576).sum()}')
print()
print(f'KBest train shape: {X_kbest_train.shape}')
print(f'KBest test shape:  {X_kbest_test.shape}')

## 7. SelectKBest Binary Detection Results

In [ ]:
rf_kbest = RandomForestClassifier(**RF_PARAMS)
result_kbest_binary = eval_binary(rf_kbest, X_kbest_train, X_kbest_test,
                                   y_bin_train, y_bin_test)

print('SelectKBest Binary Detection Results:')
print(f'  KBest (best 64 from 582): F1 = {result_kbest_binary["f1"]:.4f}')
print(f'  Config A (first 64):      F1 = 0.9835')
print(f'  Difference: {result_kbest_binary["f1"] - 0.9835:+.4f}')
print()
print(f'  Precision: {result_kbest_binary["precision"]:.4f}')
print(f'  Recall:    {result_kbest_binary["recall"]:.4f}')

## 8. SelectKBest Fault Classifier Results

In [ ]:
selector_mc = SelectKBest(f_classif, k=64)

mask_tr = (y_mc_train > 0) & (y_mc_train != jamming_class)
mask_te = (y_mc_test  > 0) & (y_mc_test  != jamming_class)
X_mc_tr, y_mc_tr = X_C_train[mask_tr], y_mc_train[mask_tr]
X_mc_te, y_mc_te = X_C_test[mask_te],  y_mc_test[mask_te]

selector_mc.fit(X_mc_tr, y_mc_tr)
X_mc_tr_kb = selector_mc.transform(X_mc_tr)
X_mc_te_kb = selector_mc.transform(X_mc_te)

unique = np.sort(np.unique(y_mc_tr))
c2i = {c: i for i, c in enumerate(unique)}
y_mc_tr_m = np.array([c2i[c] for c in y_mc_tr])
y_mc_te_m = np.array([c2i.get(c, -1) for c in y_mc_te])
valid = y_mc_te_m >= 0
X_mc_te_kb = X_mc_te_kb[valid]
y_mc_te_m  = y_mc_te_m[valid]

rf_mc_kb = RandomForestClassifier(**RF_PARAMS)
rf_mc_kb.fit(X_mc_tr_kb, y_mc_tr_m)
y_pred_kb = rf_mc_kb.predict(X_mc_te_kb)
f1_kb = f1_score(y_mc_te_m, y_pred_kb, average='macro', zero_division=0)

print('SelectKBest Fault Classifier Results:')
print(f'  KBest (best 64 from 582): F1-macro = {f1_kb:.4f}')
print(f'  Config A (first 64):      F1-macro = 0.9447')
print(f'  Difference: {f1_kb - 0.9447:+.4f}')

## 9. Complete Comparison — All Trials

In [ ]:
print('Complete Results Summary')
print('=' * 70)
print()
print('BINARY ANOMALY DETECTION (F1)')
print(f'{"Method":<35} {"Config A":>10} {"Config B":>10} {"Config C":>10}')
print('-' * 70)
print(f'{"Baseline (RF, no SMOTE)":<35} {"0.9835":>10} {"0.9708":>10} {"0.9665":>10}')
print(f'{"Best algo (LightGBM/XGBoost)":<35} {"0.9856":>10} {"0.9836":>10} {"0.9835":>10}')

a_sm = smote_binary_rf.get("A (64-dim)", {}).get("f1", 0)
b_sm = smote_binary_rf.get("B (320-dim)", {}).get("f1", 0)
c_sm = smote_binary_rf.get("C (582-dim)", {}).get("f1", 0)
print(f'{"SMOTE + RF":<35} {a_sm:>10.4f} {b_sm:>10.4f} {c_sm:>10.4f}')
print(f'{"SelectKBest (best 64 of 582)":<35} {result_kbest_binary["f1"]:>10.4f} {"—":>10} {"—":>10}')

print()
print('FAULT CLASSIFICATION (F1-macro)')
print(f'{"Method":<35} {"Config A":>10} {"Config B":>10} {"Config C":>10}')
print('-' * 70)
print(f'{"Baseline (RF, no SMOTE)":<35} {"0.9447":>10} {"0.8866":>10} {"0.8872":>10}')
print(f'{"Best algo (XGBoost/LightGBM)":<35} {"0.9564":>10} {"0.9404":>10} {"0.9045":>10}')

a_mc = smote_mc_rf.get("A (64-dim)", {}).get("f1_macro", 0)
b_mc = smote_mc_rf.get("B (320-dim)", {}).get("f1_macro", 0)
c_mc = smote_mc_rf.get("C (582-dim)", {}).get("f1_macro", 0)
print(f'{"SMOTE + RF":<35} {a_mc:>10.4f} {b_mc:>10.4f} {c_mc:>10.4f}')
print(f'{"SelectKBest (best 64 of 582)":<35} {f1_kb:>10.4f} {"—":>10} {"—":>10}')

## 10. Visualization

In [ ]:
methods = ['Baseline
RF', 'Best
Algo', 'SMOTE
+RF', 'KBest
64']

binary_A  = [0.9835, 0.9856, smote_binary_rf.get("A (64-dim)",{}).get("f1",0),
             result_kbest_binary["f1"]]
binary_B  = [0.9708, 0.9836, smote_binary_rf.get("B (320-dim)",{}).get("f1",0), None]
binary_C  = [0.9665, 0.9835, smote_binary_rf.get("C (582-dim)",{}).get("f1",0), None]

fault_A   = [0.9447, 0.9564, smote_mc_rf.get("A (64-dim)",{}).get("f1_macro",0), f1_kb]
fault_B   = [0.8866, 0.9404, smote_mc_rf.get("B (320-dim)",{}).get("f1_macro",0), None]
fault_C   = [0.8872, 0.9045, smote_mc_rf.get("C (582-dim)",{}).get("f1_macro",0), None]

x = np.arange(len(methods))
w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, title, a_vals, b_vals, c_vals, ylabel, ylim in [
    (axes[0], 'Binary Anomaly Detector', binary_A, binary_B, binary_C, 'F1 Score', (0.92, 1.01)),
    (axes[1], 'Fault Classifier',        fault_A,  fault_B,  fault_C,  'F1-macro', (0.85, 1.01)),
]:
    b1 = ax.bar(x - w, a_vals, w, label='Config A (64)', color=COLORS['normal'], edgecolor='white')
    b2 = ax.bar(x,     [v if v is not None else 0 for v in b_vals], w,
                label='Config B (320)', color=COLORS['anomaly'], edgecolor='white')
    b3 = ax.bar(x + w, [v if v is not None else 0 for v in c_vals], w,
                label='Config C (582)', color=COLORS['synthetic'], edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=9, color=COLORS['axis'])
    ax.set_ylabel(ylabel, fontsize=11, color=COLORS['axis'])
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=12, color=COLORS['axis'])
    ax.set_facecolor(COLORS['grid'])
    ax.grid(axis='y', color='white', linewidth=0.7)
    ax.spines[['top','right']].set_visible(False)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'trials_comparison.png'), dpi=150)
plt.show()

## 11. Final Decision

In [ ]:
results = {
    'Config A — Baseline RF':          {'binary': 0.9835, 'fault': 0.9447},
    'Config A — Best Algorithm':       {'binary': 0.9856, 'fault': 0.9564},
    'Config A — SMOTE + RF':           {'binary': smote_binary_rf.get("A (64-dim)",{}).get("f1",0),
                                        'fault':  smote_mc_rf.get("A (64-dim)",{}).get("f1_macro",0)},
    'Config A — SelectKBest':          {'binary': result_kbest_binary["f1"], 'fault': f1_kb},
    'Config B — Best Algorithm':       {'binary': 0.9836, 'fault': 0.9404},
    'Config B — SMOTE + RF':           {'binary': smote_binary_rf.get("B (320-dim)",{}).get("f1",0),
                                        'fault':  smote_mc_rf.get("B (320-dim)",{}).get("f1_macro",0)},
    'Config C — Best Algorithm':       {'binary': 0.9835, 'fault': 0.9045},
    'Config C — SMOTE + RF':           {'binary': smote_binary_rf.get("C (582-dim)",{}).get("f1",0),
                                        'fault':  smote_mc_rf.get("C (582-dim)",{}).get("f1_macro",0)},
}

best_binary = max(results, key=lambda k: results[k]['binary'])
best_fault  = max(results, key=lambda k: results[k]['fault'])

print('Final Decision:')
print(f'  Best Binary F1:  {best_binary} = {results[best_binary]["binary"]:.4f}')
print(f'  Best Fault F1:   {best_fault}  = {results[best_fault]["fault"]:.4f}')
print()
print('Compared to TelecomTS published baselines:')
print(f'  Mantis Binary F1:  0.800  → Your best: {results[best_binary]["binary"]:.4f} (+{results[best_binary]["binary"]-0.800:.3f})')
print(f'  Toto Fault F1:     0.848  → Your best: {results[best_fault]["fault"]:.4f} (+{results[best_fault]["fault"]-0.848:.3f})')